# Chapter 03-03 · Error bars from the data you actually have

**Label:** Core  |  **Time:** ~50 minutes  |  **Difficulty:** moderate - the technique is four lines;
what the result means takes the rest of the chapter

**Prerequisites:** 03-02. You should be able to explain a sampling distribution and a standard error.

**Position in the learning path:** module 03, chapter 3 of 8. Before: **03-02**. After: **03-04**,
which starts probability from counts.

---

## Why this matters

03-02 measured how much a number moves - by inventing a population and drawing from it a hundred
thousand times. You will never be able to do that. You get one sample, once.

This chapter gets the same answer from that one sample, using a technique you can implement in four
lines and apply to **any** statistic - a mean, a median, a ratio, a model's accuracy, the gap between
two groups - without a formula for any of them.

It is called the bootstrap, and it is the most practically useful idea in this module. It is also not
magic, and half of this chapter is spent finding out exactly where it breaks, because a technique
that quietly fails is worse than no technique.

## What you will be able to do

- Compute a confidence interval for any statistic, from one sample, in four lines
- Check a bootstrap against a known truth, and say how close it gets
- State what "95% confidence" actually means, and demonstrate it
- Name two situations where the bootstrap fails, and show one failing completely
- Report an estimate in a form that carries its own uncertainty

## Warm-up: retrieve, do not reread

1. What is the standard error of the mean, as a formula?
2. Why is the sampling distribution narrower than the population?
3. Two arms of 100 drawn from the same population differed by more than 5% how often?
4. When is the median a *more precise* estimate than the mean?

<br>

*Answers: (1) `sd / sqrt(n)`. (2) averaging dilutes extremes; a sample containing one bad value
probably also contains ordinary ones. (3) 61.6% of the time. (4) when the population has heavy tails -
five times more precise on the contaminated bus data.*

## The problem, stated precisely

Here is one week of bus mornings. Seven numbers - it is all you have.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The population is kept only so we can CHECK our answers at the end.
# The analysis below is allowed to look at `week` and nothing else.
pop_rng = np.random.default_rng(11)
population = pop_rng.gamma(2.0, 3.0, 1_000_000)
TRUE_MEAN = population.mean()

week_rng = np.random.default_rng(1)
week = population[week_rng.integers(0, len(population), 7)]

print("your data:", np.sort(week).round(2))
print("its mean :", round(float(week.mean()), 3))

The mean of your week is **4.745 minutes**. How wrong might that be?

03-02's formula, `sd / sqrt(n)`, needs the *population's* standard deviation, which you do not have.
Substituting your sample's own standard deviation is the usual move, and 03-02's E8 showed it is
biased low on small samples. There is also a deeper problem: that formula only works for a mean.
There is no equivalent formula for a median, and none at all for the sorts of things you actually
want intervals on - a model's F1 score, the ratio of two group averages, the 90th percentile.

**The bootstrap's idea.** The reason you cannot compute the sampling distribution is that you cannot
draw more samples from the population. But you have something that resembles the population:
**your sample**. So draw from *that* instead - repeatedly, with replacement, at the same size - and
watch how much the statistic moves.

It sounds like it should not work. Check whether it does.

In [ ]:
def bootstrap(sample, statistic, repeats=10_000, seed=0):
    # Resample with replacement, `repeats` times, applying `statistic` to each row.
    rng = np.random.default_rng(seed)
    resamples = sample[rng.integers(0, len(sample), (repeats, len(sample)))]
    return statistic(resamples, axis=1)


boot_means = bootstrap(week, np.mean)

print("bootstrap standard error : %.4f" % boot_means.std())
print("true standard error      : %.4f   (from 03-02, using the population)"
      % (population.std() / np.sqrt(7)))
print("the sd/sqrt(n) formula   : %.4f   (using this sample's own sd)"
      % (week.std(ddof=1) / np.sqrt(7)))

### It works, from seven numbers

The bootstrap says the standard error is **1.6483**. The true answer, which required a million-row
population and a hundred thousand simulated weeks, is **1.6048**.

That is within 3%, from resampling seven numbers.

Look at what the four lines did. `rng.integers(0, 7, (10000, 7))` builds ten thousand rows of seven
positions, drawn with replacement - so one resample might be positions `[3, 3, 0, 6, 1, 3, 5]`,
using the fourth morning three times and never using the second or fifth. Each of those rows is a
"week you might have had, if the world produced mornings like the ones you saw". Averaging each row
gives ten thousand plausible means, and their spread is the standard error.

**Why it is legitimate**, in one sentence: the sampling distribution describes how a statistic varies
when you draw `n` values from the population, and your sample is the best available stand-in for the
population - so drawing `n` values from your sample imitates the process. The imitation is only as
good as the stand-in, which is exactly where the failures later in this chapter come from.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.hist(boot_means, bins=60, color="#0072B2")
ax.axvline(week.mean(), color="black", linewidth=2)
ax.axvline(TRUE_MEAN, color="#D55E00", linewidth=2, linestyle="--")
ax.text(week.mean() + 0.1, 480, "your sample's mean 4.75", fontsize=9)
ax.text(TRUE_MEAN + 0.1, 380, "the truth 6.01", fontsize=9, color="#D55E00")
ax.set_xlabel("mean of a bootstrap resample")
ax.set_ylabel("count")
ax.set_title("Ten thousand weeks you might have had, built from the one you got")
plt.tight_layout()
plt.show()

### An important thing that histogram shows

The bootstrap distribution is centred on **your sample's mean**, 4.75, not on the truth, 6.01.

**The bootstrap estimates spread, not bias.** If your sample happened to miss the tail - as this one
did, giving 4.75 against a true 6.01 - the bootstrap cannot know. It will faithfully describe how
much a mean varies around 4.75, and it will not tell you that 4.75 is on the low side.

This is not a defect to be fixed; it is a limit to be remembered. **No method can tell you that your
sample was unlucky, using only that sample.** The bootstrap answers "how precise is my estimate",
never "is my estimate right".

## From a standard error to an interval

The bootstrap gives ten thousand plausible values. The simplest interval takes the middle 95% of them
and quotes the ends - the **percentile interval**.

In [ ]:
low, high = np.quantile(boot_means, [0.025, 0.975])
print("estimate       : %.2f minutes" % week.mean())
print("95%% interval   : %.2f to %.2f" % (low, high))
print("does it contain the true mean (%.2f)? %s" % (TRUE_MEAN, "yes" if low <= TRUE_MEAN <= high else "no"))

### What that interval means, and what it does not

**It does not mean:** "there is a 95% probability the true mean is between 2.03 and 8.28." The true
mean is a fixed number - it is 6.0064 - and it either is in that range or it is not. There is no
probability about it once the interval is computed.

**It does mean:** the *procedure* that produced this interval contains the true value 95% of the time,
across repeated samples. The 95% is a property of the method, not of the particular interval on your
screen.

That distinction sounds like pedantry. It is not, and the next section is the reason: **you can
measure whether the procedure actually delivers what it promises**, and on small samples it does not.

## Failure lab: does a 95% interval contain the truth 95% of the time?

We still have the population, so this is checkable. Draw a fresh sample, bootstrap it, build the
interval, ask whether the true mean is inside - and repeat fifteen hundred times.

**A method that works returns 0.95. Predict what you think it returns at n = 7 before running.**

In [ ]:
cover_rng = np.random.default_rng(0)

rows = []
for n in [7, 15, 30, 100, 300]:
    hits, widths = 0, []
    for _ in range(1500):
        sample = population[cover_rng.integers(0, len(population), n)]
        resampled = sample[cover_rng.integers(0, n, (1000, n))].mean(axis=1)
        lo, hi = np.quantile(resampled, [0.025, 0.975])
        widths.append(hi - lo)
        hits += lo <= TRUE_MEAN <= hi
    rows.append({"sample size n": n,
                 "actual coverage of a '95%' interval": round(hits / 1500, 3),
                 "average width": round(float(np.mean(widths)), 3)})

print(pd.DataFrame(rows).to_string(index=False))

### Diagnosis: it is too narrow when you need it most

A "95%" interval delivers **85.9% coverage at n = 7**. One time in seven it misses, not one time in
twenty. Coverage climbs to 0.933 at n = 30 and only reaches 0.948 at n = 300.

The cause is the one from 03-02's E8. The bootstrap resamples *your* seven values, so it can only
reproduce the variety those seven contain. A small sample almost always under-represents the tail -
there are more ways to miss a rare large value than to catch it - so the resamples are less varied
than reality and the interval comes out too narrow. The bootstrap inherits your sample's luck.

**What to do about it:**

- **Report the sample size next to the interval, always.** An interval from n = 7 and one from
  n = 300 are different kinds of object and the notation does not distinguish them.
- **Treat small-sample intervals as optimistic.** At n = 7, "95%" here means about 86%.
- Better interval methods exist - BCa corrects for skew and bias, and the `t` interval widens for
  small `n` - and they help, without fixing the underlying problem that seven values cannot tell you
  about a tail they did not sample.
- **Below about n = 20, the honest report is the sample itself.** Show the seven numbers.

The general lesson is worth more than the technique: **an uncertainty estimate is itself an estimate,
and it is least reliable exactly when it matters most.**

### Seeing coverage rather than reading it

Forty independent samples of thirty mornings, forty intervals, one truth.

In [ ]:
picture_rng = np.random.default_rng(12)

fig, ax = plt.subplots(figsize=(8, 5))
misses = 0
for i in range(40):
    sample = population[picture_rng.integers(0, len(population), 30)]
    resampled = sample[picture_rng.integers(0, 30, (3000, 30))].mean(axis=1)
    lo, hi = np.quantile(resampled, [0.025, 0.975])
    hit = lo <= TRUE_MEAN <= hi
    misses += not hit
    ax.plot([lo, hi], [i, i], color="#0072B2" if hit else "#D55E00", linewidth=2)
    ax.plot(sample.mean(), i, "o", color="black", markersize=3)

ax.axvline(TRUE_MEAN, color="black", linestyle="--", linewidth=1.5)
ax.set_xlabel("minutes late")
ax.set_ylabel("sample number")
ax.set_title("40 samples of 30, 40 intervals - %d missed the truth" % misses)
plt.tight_layout()
plt.show()
print("intervals containing the true mean: %d of 40" % (40 - misses))

Thirty-seven of forty - close to the 0.933 the coverage table gave for n = 30.

**This picture is what "95% confidence" means.** Each interval is either right or wrong, and you never
learn which. What you know is the fraction of such intervals, over many samples, that would be right.
The three red ones look exactly like the blue ones from the inside.

## Failure lab 2: where the bootstrap fails completely

The bootstrap works because your sample stands in for the population. That stand-in is good for the
middle of a distribution and bad at its edges - so statistics that depend on the edges break.

The clearest case is the **maximum**. Use a population with a genuine maximum so there is a right
answer: values spread uniformly between 0 and 10.

In [ ]:
uniform_rng = np.random.default_rng(4)
bounded = uniform_rng.uniform(0, 10, 1_000_000)
TRUE_MAX = 10.0

max_rng = np.random.default_rng(3)
rows = []
for n in [30, 200, 1000]:
    hits, widths = 0, []
    for _ in range(1000):
        sample = bounded[max_rng.integers(0, len(bounded), n)]
        resampled = sample[max_rng.integers(0, n, (600, n))].max(axis=1)
        lo, hi = np.quantile(resampled, [0.025, 0.975])
        widths.append(hi - lo)
        hits += lo <= TRUE_MAX <= hi
    rows.append({"sample size n": n,
                 "coverage for the MAXIMUM": round(hits / 1000, 3),
                 "average width": round(float(np.mean(widths)), 3)})

print(pd.DataFrame(rows).to_string(index=False))
print("\nthe true maximum is exactly %.1f" % TRUE_MAX)

### Diagnosis: zero coverage, and it gets worse

**The interval never contains the truth. Not rarely - never, at any sample size.** And the width
shrinks from 0.961 to 0.030 as `n` grows, so the method becomes *more confident* while remaining
completely wrong.

The reason is embarrassingly simple once you see it: **a resample cannot contain a value the sample
did not have.** The largest value in any resample is at most the largest value in your sample, which
is always below the true maximum. Every bootstrap interval for a maximum lies entirely below the
answer, by construction.

This connects directly to 03-02's E15, which found that the sample maximum does not converge on
anything. Here is the same fact in a form that bites: **a technique can be silently, completely wrong,
and its output looks exactly like an output that works.** The 0.030 width at n = 1000 is the most
dangerous number in this chapter.

**Where the bootstrap is unreliable, in order of how often you will meet it:**

| Situation | Why |
|---|---|
| Very small samples | Too little variety to resample; intervals too narrow (0.859 at n = 7) |
| Maxima, minima, extreme percentiles | Resamples cannot exceed the sample's own range |
| Heavy-tailed data | Whether a rare huge value is in your sample dominates everything |
| Dependent observations - time series, repeated measures, clustered rows | Ordinary resampling destroys the dependence and reports intervals that are far too narrow. Module 09 uses block bootstrapping instead |
| Statistics near a boundary - a proportion at 0 or 1 | The interval can include impossible values, or collapse to a point |

**Where it is excellent:** means, medians, quantiles that are not extreme, correlations, differences
between groups, ratios, and model scores - on samples of a few dozen or more, with independent rows.
That covers most of what you will need.

## The reason to learn this: it works on anything

Formulas exist for the standard error of a mean and, with effort, a few other things. The bootstrap
needs no formula at all - you change one function and everything else stays the same.

Two questions the transport authority actually has, neither of which has a standard error you could
look up.

In [ ]:
group_rng = np.random.default_rng(20)
weekday = population[group_rng.integers(0, len(population), 60)]
weekend = population[group_rng.integers(0, len(population), 40)] * 0.75   # genuinely better

print("weekday : %.3f minutes late over 60 mornings" % weekday.mean())
print("weekend : %.3f minutes late over 40 mornings" % weekend.mean())
print("observed difference: %.3f minutes" % (weekday.mean() - weekend.mean()))

In [ ]:
def bootstrap_two_groups(a, b, statistic, repeats=10_000, seed=21):
    rng = np.random.default_rng(seed)
    out = np.empty(repeats)
    for i in range(repeats):
        ra = a[rng.integers(0, len(a), len(a))]
        rb = b[rng.integers(0, len(b), len(b))]
        out[i] = statistic(ra, rb)
    return out


differences = bootstrap_two_groups(weekday, weekend, lambda a, b: a.mean() - b.mean())
ratios = bootstrap_two_groups(weekday, weekend, lambda a, b: b.mean() / a.mean(), seed=22)

lo_d, hi_d = np.quantile(differences, [0.025, 0.975])
lo_r, hi_r = np.quantile(ratios, [0.025, 0.975])

print("difference weekday - weekend : %.3f  95%% interval %.3f to %.3f" % (
    weekday.mean() - weekend.mean(), lo_d, hi_d))
print("  interval excludes zero: %s" % (lo_d > 0))
print()
print("ratio weekend / weekday      : %.4f  95%% interval %.4f to %.4f" % (
    weekend.mean() / weekday.mean(), lo_r, hi_r))
print()
print("(the true difference, which we are allowed to peek at, is %.3f)" % (TRUE_MEAN * 0.25))

### Reading these two

**The difference**, 2.19 minutes, has an interval of 1.00 to 3.42 that excludes zero - so the weekend
service really does look better, and the data is consistent with an improvement anywhere between one
minute and three and a half. The true difference is 1.50, comfortably inside.

**The ratio**, 0.624, has an interval of 0.474 to 0.811. There is no simple formula for the standard
error of a ratio of two means - it is the sort of thing that needs an approximation and a page of
algebra - and the bootstrap produced it by changing one lambda.

Notice that the ratio's interval is **not symmetric** around its estimate: 0.150 below and 0.187
above. Ratios are skewed, and a formula assuming symmetry would have got the shape wrong even if it
got the width right. The percentile bootstrap inherits the shape for free, which is one of its real
advantages over "estimate plus or minus two standard errors".

### The habit this should leave you with

Whenever you report a number computed from data, ask what its interval is. If a formula exists, use
it. If not - which is most of the time, for anything interesting - resample. Four lines is a low
enough price that "I did not have a way to compute the uncertainty" stops being a reason.

## Common misconceptions

**"The bootstrap creates data."**
It creates *rearrangements* of the data you have. It cannot add information, which is precisely why
it fails on maxima and struggles at n = 7.

**"A 95% confidence interval has a 95% chance of containing the true value."**
The true value is fixed and your interval either contains it or does not. The 95% describes the
procedure across repeated samples - and the coverage table showed the procedure delivering 85.9% at
n = 7, so even that promise needs checking.

**"A wide interval means the analysis was done badly."**
It usually means the sample is small, which is a fact about the data rather than the analyst. A
narrow interval on seven observations should worry you far more.

**"If two intervals overlap, the difference is not real."**
This is a common shortcut and it is wrong in the conservative direction - overlapping intervals can
still correspond to a clear difference. Bootstrap the **difference itself**, as above, and look at
whether *that* interval contains zero.

**"More bootstrap repeats give a better answer."**
More repeats reduce the noise in *simulating* the interval, and do nothing about the interval being
wrong. Ten thousand is plenty; a hundred thousand buys smoother quantiles and no more accuracy.

**"The bootstrap needs the data to be normal."**
It needs no distributional assumption at all - that is its main attraction. It needs *independent*
observations, and that assumption is the one people actually violate.

## Exercises

Solutions: `solutions/03_math_foundations/03-03_uncertainty_solutions.ipynb`.

### Quick understanding

**E1.** In two sentences: what does the bootstrap resample, and why is that a reasonable stand-in for
sampling from the population?

**E2.** Why is the bootstrap distribution centred on your sample's estimate rather than on the truth,
and what does that mean it cannot detect?

**E3.** State what a 95% confidence interval means, in a sentence that would survive a statistician
reading it.

### Hand calculation

**E4.** You have five values: `[2, 4, 4, 9, 11]`. Write down three possible bootstrap resamples by
hand and their means. What is the largest mean any resample could produce, and the smallest?

**E5.** A bootstrap of 10,000 resamples gives a 95% interval by taking the 250th and 9,750th values
of the sorted results. Which positions would you take for a 90% interval, and for a 99% one? Which of
the three is estimated least reliably from 10,000 resamples, and why?

### Coding

**E6.** Write `interval(sample, statistic, level=0.95)` returning the estimate and the two ends. Use
it on the week of bus data for the mean, the median, and the 90th percentile. Which is widest, and
why?

**E7.** Run the coverage experiment for the **median** at n = 7, 31 and 101. Compare with the mean's
coverage from the chapter and explain any difference.

**E8.** The chapter's coverage experiment used the percentile interval. Implement the **basic**
bootstrap interval instead - reflecting the resampled values around the estimate, `2*estimate - hi`
to `2*estimate - lo` - and compare its coverage at n = 7 and n = 30. Which is better here?

**E9.** Bootstrap a **correlation**. Generate 40 paired points with a true correlation of 0.5, then
produce a 95% interval for the correlation from the sample alone. Repeat for n = 200. How does the
interval width change, and is it symmetric?

### Interpretation

**E10.** A colleague reports "conversion improved by 0.4%, 95% CI -0.9% to 1.7%" and concludes the
feature does not work. What is wrong with that conclusion, and what would you say instead?

**E11.** Two models score 0.83 and 0.85 on the same 500-row test set. Their bootstrap intervals
overlap heavily. Explain why that is not sufficient to conclude they perform the same, and describe
the calculation you would do instead.

### Debugging

**E12.** An analyst bootstraps a time series of daily sales by resampling individual days, and reports
a very narrow interval for the weekly trend. Explain what is wrong, what direction the error goes,
and name the fix.

### Exam and interview reasoning

**E13.** "Your model has 91% accuracy. How confident are you in that number?" Answer in five
sentences, including one number you would compute and one thing you would ask about the test set.

### Transfer to a different situation

**E14.** A drug trial reports a median survival improvement of 4 months from 60 patients. Describe how
you would put an interval on that, what could make the bootstrap unreliable here, and one thing you
would check about the data first.

### Explain it to someone non-technical

**E15.** Explain the bootstrap to a manager in under 90 words, without using the words resample,
distribution or statistic.

### Optional challenge

**E16.** The bootstrap fails for the maximum. Design and test a *different* estimator of the upper
bound of a uniform distribution that does work - one that can exceed the sample maximum. Measure its
coverage at n = 30, 200 and 1000, and compare with the bootstrap's 0.000.

In [ ]:
# Your workspace. In memory: population, TRUE_MEAN, week, bootstrap,
# boot_means, bounded, weekday, weekend, bootstrap_two_groups.

## Mastery check

- [ ] Write the bootstrap in four lines from memory
- [ ] Explain why resampling your own sample estimates the sampling distribution
- [ ] State what the bootstrap estimates and what it cannot
- [ ] Say what "95%" refers to, and why the coverage table matters
- [ ] Name three statistics where the bootstrap fails, and why in each case
- [ ] Produce an interval for a difference between two groups, and read it correctly

## What should now feel instinctive

- Reaching for a resample whenever a number needs an error bar and no formula exists
- Asking the sample size before reading any interval
- Bootstrapping the difference rather than comparing two intervals by eye
- Distrusting a narrow interval on a small sample more than a wide one
- Remembering that the bootstrap describes precision, never correctness

## Flashcards

| Front | Back |
|---|---|
| The bootstrap, in one line | Resample your data with replacement, same size, recompute the statistic, look at the spread |
| What it estimates | The sampling distribution - spread only, never bias |
| Percentile interval | The 2.5th and 97.5th percentiles of the bootstrap results |
| "95% confidence" | A property of the procedure over repeated samples, not of your interval |
| Coverage at n = 7 here | 0.859, not 0.95 - small-sample intervals are too narrow |
| Bootstrap for a maximum | Coverage 0.000, and the interval narrows as n grows |
| Comparing two groups | Bootstrap the difference; do not compare two intervals by eye |
| Assumption it does need | Independent observations. Not normality |

## Next

**03-04 · Probability and conditional probability, with counts.** Everything so far has been about
summarising numbers you have. The next three chapters are about reasoning with uncertainty
directly - probability from counted tables, then Bayes' rule, then the functions and gradients that
make models fit.